In [1]:
using Plots
using LinearAlgebra

# --- 物理定数と設定 ---
const N_cutoff = 30      # 計算する基底の数（無限次元をここで打ち切る）
const x_grid = -6:0.1:6  # 空間座標（ここでは電場の強さに相当）
const dt = 0.1           # 時間刻み
const steps = 100        # アニメーションのフレーム数

# --- 波動関数の定義（調和振動子の固有関数） ---
# エルミート多項式 H_n(x) を再帰的に計算して、正規化された波動関数 φ_n(x) を返す
function harmonic_eigenstate(n, x)
    # H_0(x) = 1, H_1(x) = 2x
    if n == 0
        return exp(-x^2 / 2) / pi^0.25
    elseif n == 1
        return sqrt(2) * x * exp(-x^2 / 2) / pi^0.25
    end

    # 漸化式で計算 H_{n} = 2x H_{n-1} - 2(n-1) H_{n-2}
    # 数値安定性のため、正規化定数も含めて漸化式を回す
    val_prev2 = exp(-x^2 / 2) / pi^0.25      # n=0
    val_prev1 = sqrt(2) * x * val_prev2      # n=1
    val_curr = 0.0

    for i in 2:n
        val_curr = (sqrt(2) * x * val_prev1 - sqrt(i - 1) * val_prev2) / sqrt(i)
        val_prev2 = val_prev1
        val_prev1 = val_curr
    end
    return val_prev1
end

# 重ね合わせ状態の波動関数 Ψ(x, t) を計算
function get_wavefunction(coeffs, t, x_vals)
    psi_vals = zeros(ComplexF64, length(x_vals))
    
    # Ψ(x,t) = Σ c_n * exp(-i E_n t) * φ_n(x)
    # エネルギー E_n = n + 0.5 (ħω=1とする)
    for n in 0:(N_cutoff - 1)
        phase = exp(-im * (n + 0.5) * t)
        c_n = coeffs[n + 1]
        
        # 配列計算を避けて各点ごとに足し合わせる（高速化）
        for (i, x) in enumerate(x_vals)
            psi_vals[i] += c_n * phase * harmonic_eigenstate(n, x)
        end
    end
    return psi_vals
end

# --- 初期状態の係数 c_n を作成 ---

# 1. 光子数確定状態（フォック状態） |n=1>
# 光子が「1個」ある状態。n=1 の係数だけ 1、他は 0
coeffs_fock = zeros(ComplexF64, N_cutoff)
coeffs_fock[2] = 1.0 # Juliaは1始まりなのでインデックス2がn=1

# 2. コヒーレント状態 |α>
# α = 2.0 程度（古典的な振幅に対応）
# c_n = exp(-|α|^2/2) * α^n / sqrt(n!)
alpha = 2.0
coeffs_coherent = zeros(ComplexF64, N_cutoff)
for n in 0:(N_cutoff - 1)
    coeffs_coherent[n+1] = exp(-abs2(alpha)/2) * (alpha^n) / sqrt(factorial(big(n)))
end

# --- アニメーション生成 ---
anim = @animate for step in 0:steps
    t = step * dt
    
    # 波動関数の計算
    psi_fock = get_wavefunction(coeffs_fock, t, x_grid)
    psi_coherent = get_wavefunction(coeffs_coherent, t, x_grid)
    
    # 確率密度 |Ψ|^2
    prob_fock = abs2.(psi_fock)
    prob_coherent = abs2.(psi_coherent)

    # プロット作成
    p1 = plot(x_grid, prob_fock, 
        title="Photon Number State |n=1>", 
        label="Probability", 
        ylim=(0, 0.6), 
        lw=2, fill=(0, 0.3, :blue), linecolor=:blue,
        xlabel="Electric Field E", ylabel="|Ψ|^2")
    
    p2 = plot(x_grid, prob_coherent, 
        title="Coherent State |α=2.0>", 
        label="Probability", 
        ylim=(0, 0.6), 
        lw=2, fill=(0, 0.3, :red), linecolor=:red,
        xlabel="Electric Field E", ylabel="|Ψ|^2")

    plot(p1, p2, layout=(1, 2), size=(800, 400))
end

# GIFとして保存
gif(anim, "quantum_field_visualization.gif", fps=15)
println("アニメーションを保存しました: quantum_field_visualization.gif")

アニメーションを保存しました: quantum_field_visualization.gif


[ Info: Saved animation to /home/jovyan/work/quantum_field_visualization.gif
